# Lesson 01 — LLM، Prompt Engineering، Structured Output

**پیش‌نیاز:** Lesson 00 ✓ (GPU + پکیج‌ها)

**مدت:** ۴–۶ ساعت — این درس **عمیق** است؛ عجله نکن.

**هدف:** بعد از این درس بتوانی:
- توضیح بدهی مدل چطور متن تولید می‌کند
- prompt بنویسی که برای محصول قابل‌استفاده باشد
- خروجی را به **JSON ساختاریافته** (ویرایش بلاک) تبدیل کنی

**اتصال محصول:**
- دکمه «Rewrite with AI» → بخش ۴–۶
- دکمه «Summarize» → بخش ۶
- API ویرایش بلاک `{action, block_id, content}` → بخش ۹–۱۱

---

## نقشهٔ درس

| بخش | موضوع |
|-----|--------|
| ۱ | مدل زبانی چطور کار می‌کند؟ |
| ۲ | بارگذاری مدل |
| ۳ | Tokenizer — متن → اعداد |
| ۴ | Chat template — فرمت مکالمه |
| ۵ | حلقهٔ generation |
| ۶ | پارامترها: temperature, top_p, max_tokens |
| ۷ | System prompt و مهندسی prompt |
| ۸ | خطاهای رایج + اندازه‌گیری |
| ۹ | Structured output — JSON |
| ۱۰ | قرارداد ویرایش بلاک محصول |
| ۱۱ | تمرین‌ها |

**Runtime → GPU (T4)**. Session جدید → cell نصب را دوباره اجرا کن.

---
## بخش ۱ — مدل زبانی چطور کار می‌کند؟

### ایدهٔ اصلی: پیش‌بینی توکن بعدی

LLM (Large Language Model) یک شبکهٔ عصبی است که **توکن بعدی** را پیش‌بینی می‌کند.

```
ورودی:  "Rewrite this block: Ship MVP soon"
         ↓ tokenizer
توکن‌ها: [1234, 567, 89, ...]
         ↓ مدل (میلیاردها پارامتر)
احتمال هر توکن بعدی: P("Launch"|...) = 0.3, P("Deliver"|...) = 0.2, ...
         ↓ sampling (temperature, top_p)
توکن انتخاب‌شده → decode → "Launch the MVP..."
```

**Autoregressive:** مدل یک توکن تولید می‌کند، آن را به ورودی اضافه می‌کند، توکن بعدی را پیش‌بینی می‌کند — تا `max_new_tokens` یا EOS.

### چرا برای محصول مهم است؟

| مفهوم | اثر روی AI Workspace |
|--------|----------------------|
| Token | هزینه API، سرعت، محدودیت context |
| Temperature | rewrite ثابت vs خلاق |
| System prompt | «فقط JSON بده» vs «Here is your rewrite...» |
| Context window | چند بلاک/page را یکجا بفرستی |

### Instruct vs Base

- **Base model:** فقط ادامهٔ متن — برای chat بدون آموزش dialogue ضعیف است.
- **Instruct model** (مثل Qwen2.5-Instruct): روی دستورالعمل fine-tune شده — برای محصول ما **همیشه Instruct**.

### جریان کامل در محصول

```
کاربر بلاک را انتخاب می‌کند
    → UI متن + block_id + action (rewrite) را به API می‌فرستد
    → API system prompt + user message می‌سازد
    → مدل generate می‌کند
    → JSON parse → بلاک در DB/UI آپدیت
```

## بخش ۲ — آماده‌سازی و بارگذاری مدل

In [ ]:
import sys
import time
import json
import re
from dataclasses import dataclass
from typing import Any

import torch

assert torch.cuda.is_available(), (
    "GPU خاموش است. Runtime → Change runtime type → GPU (T4) → Restart session"
)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
print(f"VRAM: {vram_gb} GB")

In [ ]:
# session تازه → یک‌بار اجرا کن
%pip install -q "transformers>=4.44" "accelerate" "sentencepiece" "einops"

### انتخاب مدل

| گزینه | پارامتر | VRAM تقریبی | کاربرد |
|-------|---------|-------------|--------|
| **Qwen2.5-1.5B-Instruct** | 1.5B | ~3 GB | یادگیری، Colab رایگان ✓ |
| Qwen2.5-3B-Instruct | 3B | ~6 GB | کیفیت بهتر اگر GPU اجازه داد |
| API (GPT/Claude) | — | — | بعداً در production |

`torch_dtype=float16` → نصف حافظه نسبت به float32.
`device_map="auto"` → مدل روی GPU قرار می‌گیرد.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()  # inference mode — dropout خاموش
load_sec = round(time.perf_counter() - t0, 1)

device = next(model.parameters()).device
param_count = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Loaded {MODEL_ID}")
print(f"  device: {device}")
print(f"  params: {param_count:.2f}B")
print(f"  load time: {load_sec}s")

---
## بخش ۳ — Tokenizer: متن چطور به اعداد تبدیل می‌شود؟

### BPE (Byte Pair Encoding)

Tokenizer متن را به **توکن** (قطعات) تقسیم می‌کند. هر توکن یک ID عددی دارد.

- کلمهٔ رایج → یک توکن: `"hello"` → `[15339]`
- کلمهٔ نادر → چند توکن: `"workspace"` → شاید `[work, space]`
- **قانون:** ~۴ کاراکتر انگلیسی ≈ ۱ توکن (تقریبی)

### چرا مهم است؟

1. **Context window:** Qwen2.5-1.5B تا ~32K توکن — اگر prompt + history + output از این بیشتر شود، truncate یا خطا.
2. **هزینه:** APIها per-token حساب می‌کنند.
3. **سرعت:** توکن بیشتر = generation طولانی‌تر.

### تمرین ذهنی

قبل از run کردن cell بعدی حدس بزن: جملهٔ زیر چند توکن است؟

In [ ]:
SAMPLE = "Ship AI workspace MVP soon. Need pages, RAG, and browser agent."

ids = tokenizer.encode(SAMPLE)
tokens = tokenizer.convert_ids_to_tokens(ids)

print("Text:", SAMPLE)
print(f"Char count: {len(SAMPLE)}")
print(f"Token count: {len(ids)}")
print(f"Ratio chars/token: {len(SAMPLE)/len(ids):.1f}")
print("\nFirst 15 tokens:")
for tid, tok in zip(ids[:15], tokens[:15]):
    print(f"  {tid:6d}  {tok!r}")

### Decode — برگشت توکن → متن

`decode(skip_special_tokens=True)` توکن‌های ویژه مثل `<|im_start|>` را حذف می‌کند.

In [ ]:
roundtrip = tokenizer.decode(ids, skip_special_tokens=True)
print("Roundtrip OK:", roundtrip.strip() == SAMPLE.strip())
print(roundtrip)

---
## بخش ۴ — Chat Template

مدل «مکالمه» را به‌صورت یک **رشتهٔ واحد** می‌بیند. Chat template نقش‌ها را با **توکن‌های ویژه** جدا می‌کند.

### فرمت Qwen (ساده‌شده)

```
<|im_start|>system
You are an AI assistant...

<|im_start|>user
Rewrite this block...

<|im_start|>assistant
```

خط آخر = **جای تولید** — مدل از اینجا ادامه می‌دهد.

### نقش‌ها

| role | کاربرد در محصول |
|------|------------------|
| `system` | قوانین ثابت: فقط JSON، لحن، ممنوعیت‌ها |
| `user` | دستور کاربر + متن بلاک |
| `assistant` | few-shot examples (اختیاری) یا history |

**هر مدل template خودش را دارد** — همیشه `apply_chat_template` بزن، دستی `<|...|>` ننویس.

In [ ]:
BLOCK = "Ship AI workspace mvp soon. Need pages, RAG, browser agent."

messages = [
    {
        "role": "system",
        "content": "You are an AI assistant inside a Notion-like workspace. Be concise.",
    },
    {
        "role": "user",
        "content": f"Rewrite this block to be clearer:\n\n{BLOCK}",
    },
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # <|im_start|>assistant\n اضافه می‌شود
)
input_token_count = len(tokenizer.encode(prompt_text))

print("--- Full prompt (what model sees) ---")
print(prompt_text)
print(f"\n--- Input tokens: {input_token_count} ---")

---
## بخش ۵ — تابع `generate` — هستهٔ `packages/ai-core`

این تابع بعداً در [`packages/ai-core/client.py`](../../packages/ai-core/client.py) می‌رود.

### مراحل داخل تابع

1. `messages` → chat template → string
2. string → token IDs → tensor روی GPU
3. `model.generate(...)` → token IDs جدید
4. فقط **توکن‌های جدید** (نه prompt) decode شوند
5. string تمیز برگردد

In [ ]:
@dataclass
class GenResult:
    text: str
    input_tokens: int
    output_tokens: int
    latency_sec: float


def generate(
    messages: list[dict],
    *,
    temperature: float = 0.2,
    max_new_tokens: int = 256,
    top_p: float = 0.9,
    do_sample: bool | None = None,
) -> GenResult:
    """Generate assistant reply from chat messages."""
    if do_sample is None:
        do_sample = temperature > 0

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    gen_kwargs: dict[str, Any] = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        gen_kwargs["temperature"] = max(temperature, 1e-5)
        gen_kwargs["top_p"] = top_p

    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    latency = time.perf_counter() - t0

    new_ids = out[0][input_len:]
    text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    return GenResult(
        text=text,
        input_tokens=input_len,
        output_tokens=len(new_ids),
        latency_sec=round(latency, 3),
    )

In [ ]:
result = generate(messages, temperature=0.2, max_new_tokens=150)
print("--- Output ---")
print(result.text)
print(
    f"\n[input={result.input_tokens} tok, "
    f"output={result.output_tokens} tok, "
    f"{result.latency_sec}s, "
    f"{result.output_tokens/max(result.latency_sec,0.001):.1f} tok/s]"
)

### نگاه به یک قدم generation (اختیاری ولی آموزنده)

مدل برای **اولین توکن خروجی** یک بردار logits (احتمال هر توکن vocabulary) می‌دهد.

In [ ]:
inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
with torch.no_grad():
    logits = model(**inputs).logits  # shape: [batch, seq_len, vocab_size]

next_logits = logits[0, -1, :]  # آخرین position → توکن بعدی
probs = torch.softmax(next_logits, dim=-1)
top_k = 8
top_probs, top_ids = torch.topk(probs, top_k)

print("Top candidates for FIRST output token:")
for pid, p in zip(top_ids.tolist(), top_probs.tolist()):
    print(f"  {p*100:5.1f}%  {tokenizer.decode([pid])!r}")

---
## بخش ۶ — پارامترهای Generation

### Temperature

logits را «تیز» یا «نرم» می‌کند قبل از sampling.

| مقدار | رفتار | محصول |
|-------|--------|--------|
| `0` | greedy — همیشه محتمل‌ترین توکن | rewrite, JSON |
| `0.1–0.3` | کم‌ریسک، کمی تنوع | **پیش‌فرض workspace** |
| `0.7–1.0` | خلاق‌تر | brainstorming |
| `>1` | آشفته | معمولاً بد |

### top_p (nucleus sampling)

فقط از توکن‌هایی sample کن که مجموع احتمالشان ≤ `top_p` (مثلاً 0.9). tail کم‌احتمال حذف می‌شود.

### max_new_tokens

سقف توکن **خروجی** (نه ورودی). برای summarize: 64–128. برای rewrite: 128–256. برای JSON: 128 کافی است.

### do_sample

- `False` + temperature=0 → greedy deterministic
- `True` → sampling با temperature/top_p

In [ ]:
print("=== Temperature comparison (same prompt) ===\n")
for temp in [0.0, 0.2, 0.8]:
    r = generate(messages, temperature=temp, max_new_tokens=100)
    print(f"--- temp={temp} ({r.output_tokens} tok, {r.latency_sec}s) ---")
    print(r.text)
    print()

In [ ]:
summarize_msgs = [
    {"role": "system", "content": "Summarize in at most 2 sentences. No preamble."},
    {"role": "user", "content": BLOCK},
]

for max_tok in [32, 64, 256]:
    r = generate(summarize_msgs, temperature=0.2, max_new_tokens=max_tok)
    print(f"max_new_tokens={max_tok:3d} → out={r.output_tokens:3d} tok, {len(r.text)} chars")
    print(r.text[:200])
    print()

---
## بخش ۷ — System Prompt Engineering

### اصول برای AI Workspace

1. **محدودیت خروجی:** «Output ONLY the rewritten block. No quotes. No preamble.»
2. **نقش:** «You are the AI editor inside AI Workspace.»
3. **فرمت:** «Return valid JSON only» (بخش ۹)
4. **زبان:** اگر UI فارسی است، در system بگو output language

### Anti-patterns (مدل این‌ها را می‌گذارد مگر جلوگیری کنی)

- `"Here is the rewritten version:"`
- markdown code fence دور JSON
- توضیح قبل/بعد از JSON
- بلاک اصلی + rewrite با هم

### Few-shot (۱ مثال در messages)

گاهی یک جفت user/assistant کمک می‌کند — ولی token مصرف می‌کند.

In [ ]:
SYSTEM_VARIANTS = {
    "vague": "You help with text.",
    "strict": (
        "You rewrite workspace blocks. "
        "Output ONLY the rewritten text. No preamble, no quotes, no markdown fences."
    ),
    "formal": (
        "Professional editor. Rewrite in formal business English. "
        "Output only the block text, nothing else."
    ),
    "bullets": (
        "Rewrite as exactly 3 bullet points starting with '- '. "
        "No intro sentence."
    ),
}

for name, system in SYSTEM_VARIANTS.items():
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Rewrite:\n\n{BLOCK}"},
    ]
    r = generate(msgs, temperature=0.2, max_new_tokens=150)
    has_preamble = r.text.lower().startswith(("here", "sure", "the rewritten"))
    print(f"=== {name} | preamble={has_preamble} ===")
    print(r.text)
    print()

In [ ]:
# Few-shot: یک مثال قبل از درخواست واقعی
few_shot_msgs = [
    {
        "role": "system",
        "content": "Rewrite blocks concisely. Output only the new text.",
    },
    {"role": "user", "content": "Rewrite:\n\nfix bug asap"},
    {"role": "assistant", "content": "Fix the critical bug as soon as possible."},
    {"role": "user", "content": f"Rewrite:\n\n{BLOCK}"},
]

r_few = generate(few_shot_msgs, temperature=0.2, max_new_tokens=150)
r_zero = generate(
    [
        {"role": "system", "content": "Rewrite blocks concisely. Output only the new text."},
        {"role": "user", "content": f"Rewrite:\n\n{BLOCK}"},
    ],
    temperature=0.2,
    max_new_tokens=150,
)

print("Few-shot:", r_few.text)
print("\nZero-shot:", r_zero.text)
print(f"\nToken cost: few-shot input={r_few.input_tokens}, zero-shot input={r_zero.input_tokens}")

### بلاک واقعی از product spec

از [`data/samples/welcome_page.json`](../../data/samples/welcome_page.json):

In [ ]:
REAL_BLOCK = (
    "Build a Notion-like workspace where AI can edit blocks, "
    "answer from workspace knowledge, and run a browser agent "
    "that writes results back as pages."
)

product_msgs = [
    {
        "role": "system",
        "content": (
            "You are the AI inside AI Workspace. "
            "Rewrite product spec blocks clearly. Output only the new block text."
        ),
    },
    {"role": "user", "content": f"Rewrite for clarity:\n\n{REAL_BLOCK}"},
]
print(generate(product_msgs, temperature=0.2, max_new_tokens=200).text)

---
## بخش ۸ — خطاهای رایج و اندازه‌گیری

| مشکل | علت | راه‌حل |
|------|-----|--------|
| CUDA OOM | مدل بزرگ + batch | مدل کوچک‌تر، `float16`، session restart |
| خروجی طولانی | max_tokens زیاد | سقف پایین + system «be concise» |
| preamble | system ضعیف | system strict + temperature=0 |
| کند | مدل روی CPU | GPU runtime |
| truncate | prompt خیلی بلند | chunk یا خلاصهٔ context |

### VRAM بعد از load

In [ ]:
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"VRAM allocated: {alloc:.2f} GB")
    print(f"VRAM reserved:  {reserved:.2f} GB")
    print(f"Headroom (~):   {vram_gb - reserved:.2f} GB")

---
## بخش ۹ — Structured Output (JSON)

### چرا JSON؟

UI/API نمی‌تواند «متن آزاد» را parse کند. برای ویرایش بلاک نیاز داری:

```json
{
  "action": "rewrite",
  "block_id": "b4",
  "content": "Pages and blocks with AI editing..."
}
```

### actionهای MVP

| action | معنی |
|--------|------|
| `rewrite` | جایگزینی محتوای بلاک |
| `summarize` | خلاصه (ممکن است بلاک جدید بسازد) |
| `split` | یک بلاک → چند بلاک (Lesson بعدی در API) |

### تکنیک prompt

1. schema را در system بنویس
2. «JSON only, no markdown»
3. temperature=0
4. parse + retry اگر invalid

In [ ]:
BLOCK_EDIT_SCHEMA = """{
  "action": "rewrite" | "summarize" | "split",
  "block_id": "string",
  "content": "string"
}"""

JSON_SYSTEM = f"""You are the AI editor in AI Workspace.
Return ONLY valid JSON matching this schema (no markdown, no explanation):
{BLOCK_EDIT_SCHEMA}
"""

EXERCISE_BLOCK = (
    "Pages and blocks. AI rewrite and summarize. RAG over pages. "
    "Browser agent with goto/click/extract. Citations back to sources."
)

json_messages = [
    {"role": "system", "content": JSON_SYSTEM},
    {
        "role": "user",
        "content": (
            f"Rewrite block b4:\n\n{EXERCISE_BLOCK}\n\n"
            "Respond with JSON only. block_id must be b4. action must be rewrite."
        ),
    },
]

raw_json = generate(json_messages, temperature=0.0, max_new_tokens=256).text
print("--- Raw model output ---")
print(raw_json)

### Parser مقاوم — مدل گاهی ```json می‌گذارد

In [ ]:
def extract_json(text: str) -> dict:
    """Parse JSON from model output; strip markdown fences if present."""
    text = text.strip()
    # حذف ```json ... ```
    fence = re.search(r"```(?:json)?\s*(\{.*\})\s*```", text, re.DOTALL)
    if fence:
        text = fence.group(1)
    else:
        # اولین { تا آخرین }
        start, end = text.find("{"), text.rfind("}")
        if start != -1 and end != -1:
            text = text[start : end + 1]
    return json.loads(text)


def validate_block_edit(data: dict) -> dict:
    required = {"action", "block_id", "content"}
    missing = required - set(data.keys())
    if missing:
        raise ValueError(f"Missing keys: {missing}")
    if data["action"] not in ("rewrite", "summarize", "split"):
        raise ValueError(f"Invalid action: {data['action']}")
    if not isinstance(data["content"], str) or not data["content"].strip():
        raise ValueError("content must be non-empty string")
    return data


try:
    parsed = validate_block_edit(extract_json(raw_json))
    print("Parse OK:")
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
except (json.JSONDecodeError, ValueError) as e:
    print(f"Parse FAILED: {e}")
    print("→ system prompt را strict‌تر کن یا retry بزن")

### Retry loop — الگوی production

In [ ]:
def block_edit_with_retry(
    block_id: str,
    block_text: str,
    action: str = "rewrite",
    max_attempts: int = 3,
) -> dict:
    user = (
        f"{action} block {block_id}:\n\n{block_text}\n\n"
        f"JSON only. block_id={block_id!r}, action={action!r}."
    )
    msgs = [{"role": "system", "content": JSON_SYSTEM}, {"role": "user", "content": user}]

    last_err = None
    for attempt in range(1, max_attempts + 1):
        raw = generate(msgs, temperature=0.0, max_new_tokens=256).text
        try:
            return validate_block_edit(extract_json(raw))
        except (json.JSONDecodeError, ValueError) as e:
            last_err = e
            msgs.append({"role": "assistant", "content": raw})
            msgs.append(
                {
                    "role": "user",
                    "content": f"Invalid: {e}. Return ONLY valid JSON, no extra text.",
                }
            )
    raise RuntimeError(f"Failed after {max_attempts} attempts: {last_err}")


edit = block_edit_with_retry("b4", EXERCISE_BLOCK, action="rewrite")
print("Final edit object (API would apply this):")
print(json.dumps(edit, indent=2, ensure_ascii=False))

---
## بخش ۱۰ — simulate API: از prompt تا آپدیت بلاک

این همان جریانی است که در Lesson 04 در FastAPI پیاده می‌کنی.

In [ ]:
# شبیه‌سازی in-memory page (مثل apps/api/main.py)
PAGE = {
    "id": "welcome",
    "title": "Product Spec",
    "blocks": [
        {"id": "b1", "type": "heading", "content": "Core features"},
        {"id": "b4", "type": "paragraph", "content": EXERCISE_BLOCK},
    ],
}


def apply_block_edit(page: dict, edit: dict) -> dict:
    if edit["action"] == "rewrite":
        for block in page["blocks"]:
            if block["id"] == edit["block_id"]:
                block["content"] = edit["content"]
                return page
        raise KeyError(f"block_id {edit['block_id']} not found")
    raise NotImplementedError(edit["action"])


print("BEFORE b4:", PAGE["blocks"][1]["content"][:80], "...")
edit = block_edit_with_retry("b4", EXERCISE_BLOCK)
PAGE = apply_block_edit(PAGE, edit)
print("\nAFTER b4:", PAGE["blocks"][1]["content"])

---
## بخش ۱۱ — تمرین‌های Lesson 01 (اجباری)

### تمرین A — ۳ prompt برای rewrite (متن آزاد)

بلاک `EXERCISE_BLOCK` را با **۳ system/user متفاوت** rewrite کن.
هدف: یکی formal، یکی bullet، یکی ultra-short.

### تمرین B — summarize

یک prompt بنویس که خلاصهٔ ۲ جمله‌ای بدهد (متن آزاد، نه JSON).

### تمرین C — JSON summarize

همان بلاک را با `action=summarize` و `block_id=b4` JSON برگردان.

### تمرین D — مقایسه

در cell مقایسه بنویس:
- کدام prompt برای UI بهتر است؟
- JSON چند بار retry لازم داشت؟
- temperature=0 vs 0.2 برای JSON چه فرقی کرد؟

### چک‌لیست
- [ ] مدل load شد + VRAM دیدی
- [ ] token count و tok/s دیدی
- [ ] temperature و system prompt را مقایسه کردی
- [ ] JSON parse + retry کار کرد
- [ ] simulate apply_block_edit را اجرا کردی
- [ ] `docs/journal.md` آپدیت شد

**تمام شد → بگو «درس 01 تمام»**

In [ ]:
# === تمرین A: ۳ prompt — TODO را پر کن ===

prompt_a1 = [
    {"role": "system", "content": "TODO: formal rewrite, output only text"},
    {"role": "user", "content": f"Rewrite:\n\n{EXERCISE_BLOCK}"},
]
prompt_a2 = [
    {"role": "system", "content": "TODO: 3 bullets"},
    {"role": "user", "content": f"Rewrite:\n\n{EXERCISE_BLOCK}"},
]
prompt_a3 = [
    {"role": "system", "content": "TODO: max 15 words"},
    {"role": "user", "content": f"Rewrite:\n\n{EXERCISE_BLOCK}"},
]

for i, p in enumerate([prompt_a1, prompt_a2, prompt_a3], 1):
    r = generate(p, temperature=0.2, max_new_tokens=200)
    print(f"\n{'='*40}\nA{i} ({r.output_tokens} tok)\n{'='*40}")
    print(r.text)

In [ ]:
# === تمرین B: summarize (free text) ===

summarize_prompt = [
    {"role": "system", "content": "TODO: summarize in 2 sentences, no preamble"},
    {"role": "user", "content": EXERCISE_BLOCK},
]
print(generate(summarize_prompt, temperature=0.2, max_new_tokens=100).text)

In [ ]:
# === تمرین C: JSON summarize ===

summarize_json = block_edit_with_retry("b4", EXERCISE_BLOCK, action="summarize")
print(json.dumps(summarize_json, indent=2, ensure_ascii=False))

In [ ]:
MY_LESSON_01_NOTES = """
بهترین rewrite prompt: A#...
چرا: ...
JSON retries needed: ...
temperature 0 vs 0.2 for JSON: ...
surprise / bug: ...
"""
print(MY_LESSON_01_NOTES.strip() or "WARNING: notes still empty — finish exercises!")

---
## گام بعدی — Lesson 02: Dataset + LoRA

الان مدل **عمومی** است. در Lesson 02:
- ۵۰+ نمونه JSONL از کارهای workspace می‌سازی
- LoRA fine-tune می‌کنی تا JSON و لحن محصول بهتر شود

مسیر کامل: [`docs/LEARNING_PATH.md`](../../docs/LEARNING_PATH.md) — **۸ درس عمیق** (نه ۱۳ درس سطحی).